# CatanRL: Training & Evaluation

Train a MaskablePPO agent for Settlers of Catan with optional Potential-Based Reward Shaping (PBRS).
Works on both Google Colab (GPU recommended) and local machines.

**Setup order:** Run cells top-to-bottom. Skip Section 4 (Evaluate Against Stronger Opponents) if you haven't trained yet — use a saved model path instead.

## 1. Setup

In [ ]:
# Install dependencies
!pip install catanatron catanatron_gym sb3-contrib stable-baselines3 gymnasium pyyaml torch numpy matplotlib tensorboard

# Clone and install the project (Colab only)
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    REPO = 'IDL-CatanRL'
    if not os.path.exists(REPO):
        !git clone https://github.com/YOUR_USERNAME/IDL-CatanRL.git
    os.chdir(REPO)
    !pip install -e . -q
    print(f'Working directory: {os.getcwd()}')
else:
    # Local: run from repo root
    !pip install -e . -q
    print(f'Working directory: {os.getcwd()}')

## 2. Configuration

Edit the variables below. `CONFIG_PATH` selects a YAML file from `configs/`; `OVERRIDES` lets you override individual keys without editing the file.

Available configs:
- `configs/pbrs_4p_weighted.yaml` — PBRS reward, 4-player vs WeightedRandom (default)
- `configs/sparse_4p_weighted.yaml` — Sparse reward baseline, same opponents
- `configs/pbrs_1v1_alphabeta.yaml` — PBRS 1v1 vs AlphaBeta

Key config options (see CLAUDE.md for full reference):

| Key | What it does |
|-----|--------------|
| `reward_type` | `"sparse"` or `"pbrs"` |
| `pbrs_lambda` | PBRS shaping strength (0 = no shaping, 1 = full) |
| `enemies` | List of opponents: `Random`, `WeightedRandom`, `VictoryPoint`, `ValueFunction`, `AlphaBeta` |
| `total_timesteps` | Total env steps to train |
| `net_arch` | Hidden layer sizes, e.g. `[256,256]` |
| `save_path` | Where the trained model is saved (`.zip` appended automatically) |

In [ ]:
CONFIG_PATH = "configs/pbrs_4p_weighted.yaml"

OVERRIDES = [
    # Uncomment and edit to override specific keys:
    # "total_timesteps=2000000",
    # "pbrs_lambda=0.5",
    # "save_path=models/my_experiment",
    # "enemies=['WeightedRandom','WeightedRandom','WeightedRandom']",
]

from catanrl.config import load_config
config = load_config(CONFIG_PATH, OVERRIDES if OVERRIDES else None)

print("Active config:")
for k, v in sorted(config.items()):
    print(f"  {k}: {v}")

## 3. Train

In [ ]:
from catanrl.train import train

model, callback = train(config)

## 3b. Visualize Training

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rewards = callback.episode_rewards
lengths = callback.episode_lengths
# episode_won is a bool list — correct even with PBRS (total reward is negative for wins
# when shaping is active, so we never use ep_reward > 0 to detect wins)
won = [1 if w else 0 for w in callback.episode_won]
window = 50

if len(rewards) >= window:
    smoothed_r = np.convolve(rewards, np.ones(window)/window, mode='valid')
    smoothed_l = np.convolve(lengths, np.ones(window)/window, mode='valid')
    win_rate   = np.convolve(won,     np.ones(window)/window, mode='valid')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(smoothed_r)
    axes[0].set_title('Episode Reward (50-ep rolling avg)')
    axes[0].set_xlabel('Episode')
    axes[0].set_ylabel('Reward')
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

    axes[1].plot(smoothed_l)
    axes[1].set_title('Episode Length (50-ep rolling avg)')
    axes[1].set_xlabel('Episode')
    axes[1].set_ylabel('Steps')

    num_players = 1 + len(config['enemies'])
    random_chance = 100.0 / num_players
    axes[2].plot(100 * win_rate)
    axes[2].set_title('Win Rate (50-ep rolling avg)')
    axes[2].set_xlabel('Episode')
    axes[2].set_ylabel('Win Rate %')
    axes[2].axhline(y=random_chance, color='red', linestyle='--', alpha=0.5,
                    label=f'{random_chance:.0f}% (random chance)')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    final_wr = 100 * np.mean(won[-window:])
    print(f'Final {window}-ep win rate: {final_wr:.1f}%  (random chance: {random_chance:.0f}%)')
else:
    print(f'Only {len(rewards)} episodes recorded, need {window} to plot')

## 4. Evaluate (same opponents as training)

In [ ]:
from catanrl.evaluate import evaluate_gym, evaluate_game_api, needs_game_api

model_path = config['save_path']

if needs_game_api(config):
    evaluate_game_api(config, model_path)
else:
    evaluate_gym(config, model_path)

## 5. Evaluate Against Stronger Opponents

Test the trained model against `ValueFunction` or `AlphaBeta` players.

**Important**: The model must have been trained with the same number of players (observation space depends on player count). A 4-player model requires exactly 3 opponents.

In [ ]:
from catanrl.evaluate import evaluate_game_api
from catanrl.config import load_config

# Path to the model you want to evaluate (from training above, or a saved .zip)
MODEL_PATH = config['save_path']   # or e.g. "models/pbrs_4p_weighted"

# --- vs AlphaBeta + 2x WeightedRandom ---
eval_config = load_config(CONFIG_PATH, [
    "enemies=['AlphaBeta','WeightedRandom','WeightedRandom']",
    "eval_games=100",  # AlphaBeta is slow; use fewer games
])
print("Evaluating vs AlphaBeta + 2x WeightedRandom...")
evaluate_game_api(eval_config, MODEL_PATH)

In [ ]:
# --- vs 3x AlphaBeta (hardest) ---
eval_config_hard = load_config(CONFIG_PATH, [
    "enemies=['AlphaBeta','AlphaBeta','AlphaBeta']",
    "eval_games=50",
])
print("Evaluating vs 3x AlphaBeta...")
evaluate_game_api(eval_config_hard, MODEL_PATH)

In [ ]:
# --- vs ValueFunction + 2x WeightedRandom ---
eval_config_vf = load_config(CONFIG_PATH, [
    "enemies=['ValueFunction','WeightedRandom','WeightedRandom']",
    "eval_games=100",
])
print("Evaluating vs ValueFunction + 2x WeightedRandom...")
evaluate_game_api(eval_config_vf, MODEL_PATH)

## 6. Lambda Ablation

Train with different PBRS lambda values to measure the effect of reward shaping strength.
λ=0 is equivalent to sparse reward; λ=1 is full PBRS.

In [ ]:
from catanrl.config import load_config
from catanrl.train import train
from catanrl.evaluate import evaluate_gym, evaluate_game_api, needs_game_api
import numpy as np

LAMBDAS = [0.0, 0.1, 0.5, 1.0]
ablation_results = {}

for lam in LAMBDAS:
    print(f"\n{'#'*60}")
    print(f"  Lambda = {lam}")
    print(f"{'#'*60}")

    ablation_config = load_config(CONFIG_PATH, [
        f"pbrs_lambda={lam}",
        f"save_path=models/ablation_lam_{lam}",
        f"tb_log=logs/ablation_lam_{lam}",
        "reward_type=pbrs",
    ])

    model, callback = train(ablation_config)
    ablation_results[lam] = callback

    if needs_game_api(ablation_config):
        evaluate_game_api(ablation_config, ablation_config['save_path'])
    else:
        evaluate_gym(ablation_config, ablation_config['save_path'])

In [ ]:
# Plot ablation results
import matplotlib.pyplot as plt
import numpy as np

window = 50
fig, ax = plt.subplots(figsize=(10, 5))

num_players = 1 + len(config['enemies'])
random_chance = 100.0 / num_players
ax.axhline(y=random_chance, color='gray', linestyle='--', alpha=0.5,
           label=f'{random_chance:.0f}% random chance')

for lam, cb in ablation_results.items():
    won = [1 if w else 0 for w in cb.episode_won]
    if len(won) >= window:
        wr = np.convolve(won, np.ones(window)/window, mode='valid')
        ax.plot(100 * wr, label=f'λ={lam}')

ax.set_title('Win Rate by Lambda (50-ep rolling avg)')
ax.set_xlabel('Episode')
ax.set_ylabel('Win Rate %')
ax.legend()
plt.tight_layout()
plt.show()